# Protocol v3 Redesign - Joint Viability Monte Carlo Study

This notebook presents a focused Monte Carlo evaluation of the latest v3 redesign.

## Objective

Identify which `yield_split` values (from `0.00` to `0.20`) generate **jointly profitable outcomes** for both protocol sides:

- LP hedger: mean return > 0
- RT underwriter: mean return > 0

## Scope

- Product geometry fixed by design (`cover=1`, barrier equals lower range bound, width = `+-7.5%`).
- Real SOL data fetched from Birdeye (hard-fail if unavailable).
- Monte Carlo built from historical weekly block bootstrap.
- No superiority claim vs alternative hedges in this notebook.

In [ ]:
import os, time
from pathlib import Path
import numpy as np
import pandas as pd
import requests
from scipy.special import ndtr
from numpy.polynomial.hermite import hermgauss

# Reproducibility
rng = np.random.default_rng(42)

# Output directory
DATA_DIR = Path('/home/sowelo/Scrivania/LiquidityHedge/lh-protocol-v3/notebooks/data')
DATA_DIR.mkdir(parents=True, exist_ok=True)
print('DATA_DIR:', DATA_DIR)

In [ ]:
# Birdeye real-data loader (strict hard-fail)
# Prefer OS environment; if missing, load from repo .env (walk upward from CWD to find lh-protocol/.env).


def _load_env_file(path: Path, overwrite: bool = False) -> None:
    if not path.is_file():
        return
    for line in path.read_text(encoding='utf-8', errors='ignore').splitlines():
        s = line.strip()
        if not s or s.startswith('#') or '=' not in s:
            continue
        k, _, v = s.partition('=')
        k, v = k.strip(), v.strip().strip('"').strip("'")
        if not k:
            continue
        if overwrite or (k not in os.environ) or (not os.getenv(k, '').strip()):
            os.environ[k] = v


def _find_repo_root() -> Path | None:
    p = Path.cwd().resolve()
    for _ in range(10):
        if (p / 'lh-protocol' / '.env').is_file():
            return p
        if p.parent == p:
            break
        p = p.parent
    return None


_repo = _find_repo_root()
_env_paths = []
if _repo is not None:
    _env_paths.extend([_repo / 'lh-protocol' / '.env', _repo / '.env'])
_env_paths.append(Path.cwd() / 'lh-protocol' / '.env')
for _p in _env_paths:
    _load_env_file(_p, overwrite=False)

BIRDEYE_API_KEY = os.getenv('BIRDEYE_API_KEY', '').strip()
SOL_MINT = 'So11111111111111111111111111111111111111112'

if not BIRDEYE_API_KEY:
    raise RuntimeError(
        'BIRDEYE_API_KEY is missing. Set it in the environment or in lh-protocol/.env at the repo root.'
    )

def fetch_birdeye_daily_sol(days=1150, chunk_days=90):
    now = int(time.time())
    start = now - int(days) * 24 * 3600
    rows = []
    url = 'https://public-api.birdeye.so/defi/ohlcv'
    headers = {'X-API-KEY': BIRDEYE_API_KEY, 'Accept': 'application/json', 'x-chain': 'solana'}
    cur = start
    while cur < now:
        nxt = min(cur + int(chunk_days) * 24 * 3600, now)
        params = {'address': SOL_MINT, 'type': '1D', 'time_from': int(cur), 'time_to': int(nxt)}
        r = requests.get(url, params=params, headers=headers, timeout=20)
        r.raise_for_status()
        js = r.json()
        if not bool(js.get('success', False)):
            raise RuntimeError(f'Birdeye request failed for [{cur}, {nxt}]')
        items = js.get('data', {}).get('items', [])
        if not isinstance(items, list):
            raise RuntimeError('Unexpected Birdeye schema: data.items is not list')
        rows.extend(items)
        cur = nxt
    if len(rows) == 0:
        raise RuntimeError('Birdeye returned empty OHLCV items')

    df = pd.DataFrame(rows)
    ts_col = 'unixTime' if 'unixTime' in df.columns else 'startUnixTime'
    close_col = 'c' if 'c' in df.columns else 'close'
    if ts_col not in df.columns or close_col not in df.columns:
        raise RuntimeError(f'Unexpected Birdeye columns: {list(df.columns)}')

    out = pd.DataFrame({
        'open_time': pd.to_datetime(df[ts_col], unit='s', utc=True),
        'close': pd.to_numeric(df[close_col], errors='coerce')
    }).dropna().drop_duplicates('open_time').sort_values('open_time').reset_index(drop=True)

    if len(out) < 365:
        raise RuntimeError(f'Birdeye returned too few daily points ({len(out)})')
    return out

hist = fetch_birdeye_daily_sol(days=1150, chunk_days=90)
closes = hist['close'].to_numpy(float)
print('Source: birdeye_SOL_daily')
print('Data points:', len(closes))

In [ ]:
# Core model functions (v3 redesign)
WIDTH = 0.075
T_WEEK = 7/365
N_LIQ = 10_000.0
PROTOCOL_FEE_RATE = 0.002
MARKUP_FLOOR = 0.99
QUAD_N = 48
GH_X, GH_W = hermgauss(QUAD_N)


def cl_value(S, L, p_l, p_u):
    S = np.asarray(S, float)
    sp, spl, spu = np.sqrt(np.clip(S, 1e-12, None)), np.sqrt(p_l), np.sqrt(p_u)
    below = S <= p_l
    above = S >= p_u
    a = np.where(below, L * (spu - spl) / (spl * spu), np.where(above, 0.0, L * (spu - sp) / (sp * spu)))
    b = np.where(below, 0.0, np.where(above, L * (spu - spl), L * (sp - spl)))
    return a * S + b


def make_pos(S0, lev=1.0):
    p_l, p_u = S0 * (1 - WIDTH), S0 * (1 + WIDTH)
    B = p_l
    L = N_LIQ * lev
    V0 = float(cl_value(np.array([S0]), L, p_l, p_u)[0])
    Vb = float(cl_value(np.array([B]), L, p_l, p_u)[0])
    return {'p_l': p_l, 'p_u': p_u, 'B': B, 'L': L, 'V0': V0, 'cap': max(0.0, V0 - Vb)}


def corridor_payoff(S_T, S0, pos):
    V_eff = cl_value(np.maximum(S_T, pos['B']), pos['L'], pos['p_l'], pos['p_u'])
    raw = np.maximum(0.0, np.where(S_T >= S0, 0.0, pos['V0'] - V_eff))
    return np.minimum(pos['cap'], raw)


def fv_quadrature(S0, sigma, pos):
    S_T = S0 * np.exp(-0.5 * sigma**2 * T_WEEK + sigma * np.sqrt(T_WEEK) * GH_X * np.sqrt(2))
    pay = corridor_payoff(S_T, S0, pos)
    return max(0.0, float(np.sum(GH_W * pay) / np.sqrt(np.pi)))


def iv_rv_regime_switching(s7, s30):
    ratio = float(s7) / max(float(s30), 1e-9)
    IVRV_MIN, IVRV_MAX = 0.99, 1.12
    frac = np.clip((ratio - 0.90) / (1.30 - 0.90), 0.0, 1.0)
    return float(IVRV_MIN + (IVRV_MAX - IVRV_MIN) * frac)


RV_ANCHOR = float(np.clip(np.std(np.diff(np.log(closes)), ddof=1) * np.sqrt(365), 0.10, 3.0))
blocks = np.array([closes[i:i+8] for i in range(len(closes)-7)], dtype=float)
print('Weekly blocks:', len(blocks))

In [ ]:
def sample_weeks(n_weeks=52, rng_local=None):
    rng_eff = rng_local if rng_local is not None else rng
    idx = rng_eff.integers(0, len(blocks), size=n_weeks)
    return blocks[idx]


def simulate_protocol_mc(y_split, fee_day, n_paths=500, n_weeks=52, lev=1.0, tx_bps_open=2.0, tx_bps_settle=2.0):
    lp_w, rt_w = [], []
    exec_cost_rate = (tx_bps_open + tx_bps_settle) / 10_000
    seed = int((round(float(y_split), 3) * 1e6 + round(float(fee_day), 7) * 1e6) * 1000) % 2**32
    rng_local = np.random.default_rng(seed)

    for _ in range(n_paths):
        rt_cap = 2_000_000.0
        idle_yield_week = (1 + 0.002)**7 - 1

        for wk in sample_weeks(n_weeks=n_weeks, rng_local=rng_local):
            S0, ST = float(wk[0]), float(wk[-1])
            lr = np.diff(np.log(wk))
            rv = float(np.clip(np.std(lr, ddof=1) * np.sqrt(365), 0.25, 1.8))

            pos = make_pos(S0, lev=lev)
            if pos['cap'] <= 1e-9:
                continue

            fv = fv_quadrature(S0, rv, pos)
            m_vol = max(MARKUP_FLOOR, iv_rv_regime_switching(rv, RV_ANCHOR))

            t_days = np.arange(1, 8) / 365.0
            sigma = max(rv, 1e-9)
            mu = -0.5 * sigma**2 * t_days
            sd = sigma * np.sqrt(t_days)
            a_log, b_log = np.log(1.0 - WIDTH), np.log(1.0 + WIDTH)
            p_in = ndtr((b_log - mu) / sd) - ndtr((a_log - mu) / sd)
            exp_fees = pos['V0'] * fee_day * 7 * float(np.mean(p_in))

            premium = max(0.0, fv * m_vol - y_split * exp_fees)
            realized_fee_day = float(np.clip(rng_local.normal(fee_day, 0.0007), 0.0003, 0.02))
            in_range_frac = float(np.mean((wk[1:] >= pos['p_l']) & (wk[1:] <= pos['p_u'])))
            fees = pos['V0'] * realized_fee_day * 7 * in_range_frac

            payoff = float(corridor_payoff(np.array([ST]), S0, pos)[0])
            Vend = float(cl_value(np.array([ST]), pos['L'], pos['p_l'], pos['p_u'])[0])
            protocol_exec_cost = exec_cost_rate * pos['V0']

            lp_pnl = (Vend - pos['V0']) + fees * (1 - y_split) + payoff - premium - protocol_exec_cost
            rt_pnl = premium * (1 - PROTOCOL_FEE_RATE) + fees * y_split - payoff

            lp_ret = lp_pnl / max(pos['V0'], 1e-9)
            rt_ret = rt_pnl / max(rt_cap, 1e-9)
            rt_cap *= (1 + rt_ret + idle_yield_week)

            lp_w.append(lp_ret)
            rt_w.append(rt_ret)

    return np.array(lp_w, float), np.array(rt_w, float)

In [ ]:
# Yield-split sweep: 0.00 -> 0.20
yield_split_grid = np.round(np.linspace(0.00, 0.20, 21), 2)

# Fixed scenario baseline for fee environment (daily LP fee yield)
# Chosen as a representative central level used in prior v3 analysis.
fee_day_baseline = 0.0023

rows = []
for ys in yield_split_grid:
    lp, rt = simulate_protocol_mc(y_split=float(ys), fee_day=float(fee_day_baseline), n_paths=500, n_weeks=52, lev=1.0)
    lp_mean = float(np.mean(lp)) if len(lp) else np.nan
    rt_mean = float(np.mean(rt)) if len(rt) else np.nan
    rows.append({
        'yield_split': float(ys),
        'lp_mean': lp_mean,
        'rt_mean': rt_mean,
        'joint_profitable': bool((lp_mean > 0.0) and (rt_mean > 0.0)) if np.isfinite(lp_mean) and np.isfinite(rt_mean) else False,
        'lp_median': float(np.median(lp)) if len(lp) else np.nan,
        'rt_median': float(np.median(rt)) if len(rt) else np.nan,
        'n_obs_lp': int(len(lp)),
        'n_obs_rt': int(len(rt)),
    })

res = pd.DataFrame(rows)
res.to_csv(DATA_DIR / 'v3_mc_joint_viability_by_yield_split.csv', index=False)

joint = res[res['joint_profitable']].copy()
print('Saved:', DATA_DIR / 'v3_mc_joint_viability_by_yield_split.csv')
print('\n--- Full yield_split results ---')
print(res.to_string(index=False))
print('\n--- Jointly profitable region ---')
if len(joint):
    print(joint.to_string(index=False))
else:
    print('No jointly profitable yield_split found at this fee baseline.')

res

## Notes for interpretation

- `joint_profitable=True` means both LP and RT mean returns are positive in the same Monte Carlo setup.
- This notebook is intentionally focused on **joint viability region discovery** for v3 redesign.
- It does not include claims of superiority versus other hedging strategies.
- If desired, a second section can add stress scenarios over multiple `fee_day` baselines to check robustness of the jointly profitable region.